# Notebook 01: Data Assembly

## Building a Country-Month Panel for Causal Inference

This notebook downloads, cleans, and merges five data sources into a single country-month panel covering African countries from 2000 to 2022. Each row in the final dataset represents one country in one month, with columns for:

- **Outcome**: civilian violence events and fatalities (UCDP GED)
- **Treatment**: UN peacekeeping mission presence (manually constructed from UN records)
- **Confounders**: GDP per capita, population, ethnic fractionalization, active armed conflict

The panel structure is essential for causal inference. It lets us observe the same country before and after peacekeeping deployment, controlling for time-invariant country characteristics.

See `docs/01_data_assembly_guide.md` for the full conceptual explanation of each data source and the merging logic.

## 1. Setup and Imports

We set the working directory to the project root using an absolute path. This prevents errors when re-running cells, a lesson from Project 1. All file paths in this notebook are relative to this root.

In [24]:
import os
os.chdir("/Users/cansezgin/Library/Mobile Documents/com~apple~CloudDocs/Claude_Projects/Computational Social Sciences/conflict-causal-inference")

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Working directory:", os.getcwd())
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

Working directory: /Users/cansezgin/Library/Mobile Documents/com~apple~CloudDocs/Claude_Projects/Computational Social Sciences/conflict-causal-inference
pandas version: 3.0.1
numpy version: 2.4.2


## 2. Outcome Variable: UCDP GED (Civilian Violence)

The UCDP Georeferenced Event Dataset records individual conflict events with coordinates, dates, actors, and fatality estimates. We filter to African countries and aggregate to country-month level.

The primary outcome is **one-sided violence** (type_of_violence == 3), defined as violence by an organized armed group deliberately targeting civilians. This is what UN peacekeeping is designed to prevent.

We create three outcome variables:
- `onesided_events`: count of one-sided violence events per country-month
- `onesided_deaths`: fatalities from one-sided violence (using the "best" estimate)
- `civilian_deaths`: civilian deaths across all violence types

### 2.1 Load and Filter UCDP GED to Africa

We load the full GED dataset and filter to African countries. UCDP uses a numeric region code: `region == 4` means Africa. We also parse the date column and extract year and month for aggregation.

In [25]:
# Load UCDP GED
ged = pd.read_csv("data/raw/GEDEvent_v25_1.csv")
print(f"Full GED dataset: {len(ged):,} events")

# Filter to Africa
ged_africa = ged[ged['region'] == 'Africa'].copy()
print(f"Africa only: {len(ged_africa):,} events")

# Parse dates and extract year/month
ged_africa['date_start'] = pd.to_datetime(ged_africa['date_start'])
ged_africa['year'] = ged_africa['date_start'].dt.year
ged_africa['month'] = ged_africa['date_start'].dt.month

# Filter to 2000-2022
ged_africa = ged_africa[(ged_africa['year'] >= 2000) & (ged_africa['year'] <= 2022)]
print(f"Africa 2000-2022: {len(ged_africa):,} events")

# Check violence types
print("\nViolence types in Africa:")
type_labels = {1: 'State-based', 2: 'Non-state', 3: 'One-sided'}
for vtype, count in ged_africa['type_of_violence'].value_counts().sort_index().items():
    print(f"  {vtype} ({type_labels[vtype]}): {count:,}")

Full GED dataset: 385,918 events
Africa only: 68,354 events
Africa 2000-2022: 43,285 events

Violence types in Africa:
  1 (State-based): 22,337
  2 (Non-state): 6,399
  3 (One-sided): 14,549


### 2.2 Standardize Country Names to ISO3 Codes

Different datasets identify countries differently. UCDP uses country names, the World Bank uses ISO3 codes (like ETH for Ethiopia), and EPR uses its own system. To merge everything cleanly, we standardize all datasets to ISO3 alpha-3 codes.

We build a mapping from UCDP country names to ISO3 codes using the `pycountry` package, with manual corrections for names that do not match automatically (e.g., "DR Congo" vs. "Congo, The Democratic Republic of the").

### 2.3 Map UCDP Country Names to ISO3 Codes

UCDP uses non-standard country names for several African states. For example, "DR Congo (Zaire)" instead of "Democratic Republic of the Congo," or "Ivory Coast" instead of "Côte d'Ivoire." We build a manual mapping dictionary covering all 43 countries. This is more reliable than automated matching, which often fails on these edge cases.

In [26]:
# Manual mapping: UCDP country name -> ISO3 code
ucdp_to_iso3 = {
    'Algeria': 'DZA',
    'Angola': 'AGO',
    'Benin': 'BEN',
    'Burkina Faso': 'BFA',
    'Burundi': 'BDI',
    'Cameroon': 'CMR',
    'Central African Republic': 'CAF',
    'Chad': 'TCD',
    'Congo': 'COG',
    'DR Congo (Zaire)': 'COD',
    'Djibouti': 'DJI',
    'Eritrea': 'ERI',
    'Ethiopia': 'ETH',
    'Gambia': 'GMB',
    'Ghana': 'GHA',
    'Guinea': 'GIN',
    'Guinea-Bissau': 'GNB',
    'Ivory Coast': 'CIV',
    'Kenya': 'KEN',
    'Kingdom of eSwatini (Swaziland)': 'SWZ',
    'Liberia': 'LBR',
    'Libya': 'LBY',
    'Madagascar (Malagasy)': 'MDG',
    'Mali': 'MLI',
    'Mauritania': 'MRT',
    'Morocco': 'MAR',
    'Mozambique': 'MOZ',
    'Namibia': 'NAM',
    'Niger': 'NER',
    'Nigeria': 'NGA',
    'Rwanda': 'RWA',
    'Senegal': 'SEN',
    'Sierra Leone': 'SLE',
    'Somalia': 'SOM',
    'South Africa': 'ZAF',
    'South Sudan': 'SSD',
    'Sudan': 'SDN',
    'Tanzania': 'TZA',
    'Togo': 'TGO',
    'Tunisia': 'TUN',
    'Uganda': 'UGA',
    'Zambia': 'ZMB',
    'Zimbabwe (Rhodesia)': 'ZWE'
}

# Apply mapping
ged_africa['iso3'] = ged_africa['country'].map(ucdp_to_iso3)

# Check for unmapped countries
unmapped = ged_africa[ged_africa['iso3'].isna()]['country'].unique()
if len(unmapped) > 0:
    print(f"WARNING: {len(unmapped)} unmapped countries: {unmapped}")
else:
    print(f"All 43 countries mapped successfully to ISO3 codes")
    
# Show a few examples
print("\nSample mapping:")
print(ged_africa[['country', 'iso3']].drop_duplicates().head(10).to_string(index=False))

All 43 countries mapped successfully to ISO3 codes

Sample mapping:
                 country iso3
                 Algeria  DZA
                  Angola  AGO
                   Benin  BEN
            Burkina Faso  BFA
                 Burundi  BDI
                Cameroon  CMR
Central African Republic  CAF
                    Chad  TCD
                   Congo  COG
                Djibouti  DJI


### 2.4 Aggregate to Country-Month Level

We collapse the event-level data into country-month counts. For each country in each month, we count:

- One-sided violence events (type_of_violence == 3) and their fatalities
- Civilian deaths across all violence types

Most country-months will have zero events. This is expected and correct. Countries without active conflict have long stretches of zeros, and we need those zeros in the panel to avoid selection bias (only analyzing months when violence happens).

In [27]:
# One-sided violence: type_of_violence == 3
onesided = ged_africa[ged_africa['type_of_violence'] == 3].copy()

# Aggregate one-sided violence to country-month
onesided_agg = (onesided
    .groupby(['iso3', 'year', 'month'])
    .agg(
        onesided_events=('id', 'count'),
        onesided_deaths=('best', 'sum')
    )
    .reset_index()
)

print(f"Country-months with one-sided violence: {len(onesided_agg):,}")
print(f"Total one-sided events: {onesided_agg['onesided_events'].sum():,}")
print(f"Total one-sided deaths: {onesided_agg['onesided_deaths'].sum():,}")

# Civilian deaths across ALL violence types
civilian_agg = (ged_africa
    .groupby(['iso3', 'year', 'month'])
    .agg(
        civilian_deaths=('deaths_civilians', 'sum'),
        all_events=('id', 'count')
    )
    .reset_index()
)

print(f"\nCountry-months with any violence: {len(civilian_agg):,}")
print(f"Total civilian deaths (all types): {civilian_agg['civilian_deaths'].sum():,}")

Country-months with one-sided violence: 2,563
Total one-sided events: 14,549
Total one-sided deaths: 112,459

Country-months with any violence: 3,835
Total civilian deaths (all types): 130,611


## 3. Build the Country-Month Skeleton

The skeleton is a complete grid: every African country gets a row for every month from January 2000 to December 2022. This is 43 countries × 276 months = 11,868 rows.

Why do we need this? Without the skeleton, we only have rows for country-months where violence occurred. That would mean our dataset only contains "bad months," creating severe selection bias. The zeros matter. A country-month with no violence and no peacekeeping is an observation. A country-month with peacekeeping and no violence is evidence that peacekeeping might be working. We need both in the data.

In [28]:
# All unique ISO3 codes from our UCDP Africa data
iso3_list = sorted(ged_africa['iso3'].unique())

# All months from Jan 2000 to Dec 2022
months = pd.date_range(start='2000-01-01', end='2022-12-01', freq='MS')

# Create the complete skeleton
skeleton = pd.DataFrame(
    [(iso3, d.year, d.month) for iso3 in iso3_list for d in months],
    columns=['iso3', 'year', 'month']
)

# Add a date column for plotting
skeleton['date'] = pd.to_datetime(skeleton[['year', 'month']].assign(day=1))

# Add country names back
iso3_to_name = {v: k for k, v in ucdp_to_iso3.items()}
skeleton['country_name'] = skeleton['iso3'].map(iso3_to_name)

print(f"Skeleton: {len(skeleton):,} country-months")
print(f"Countries: {skeleton['iso3'].nunique()}")
print(f"Date range: {skeleton['date'].min().strftime('%Y-%m')} to {skeleton['date'].max().strftime('%Y-%m')}")
print(f"Months per country: {len(months)}")
print(f"\nExpected: 43 × 276 = {43 * 276:,}")

Skeleton: 11,868 country-months
Countries: 43
Date range: 2000-01 to 2022-12
Months per country: 276

Expected: 43 × 276 = 11,868


### 3.1 Merge Outcome Variables onto Skeleton

We use left joins from the skeleton so every country-month row is preserved. Country-months with no violence events get filled with zeros. This is the correct value, not a missing value. No recorded one-sided violence event means zero events happened that month.

In [29]:
# Merge one-sided violence counts
panel = skeleton.merge(onesided_agg, on=['iso3', 'year', 'month'], how='left')

# Merge civilian deaths from all violence types
panel = panel.merge(civilian_agg, on=['iso3', 'year', 'month'], how='left')

# Fill NaN with 0: no events recorded means zero events
for col in ['onesided_events', 'onesided_deaths', 'civilian_deaths', 'all_events']:
    panel[col] = panel[col].fillna(0).astype(int)

# Verify
print(f"Panel rows: {len(panel):,} (should be 11,868)")
print(f"\nOutcome summary:")
print(f"  Country-months with zero one-sided events: {(panel['onesided_events'] == 0).sum():,} ({(panel['onesided_events'] == 0).mean():.1%})")
print(f"  Country-months with 1+ one-sided events:   {(panel['onesided_events'] > 0).sum():,} ({(panel['onesided_events'] > 0).mean():.1%})")
print(f"\n  Total one-sided events: {panel['onesided_events'].sum():,}")
print(f"  Total one-sided deaths: {panel['onesided_deaths'].sum():,}")
print(f"  Total civilian deaths:  {panel['civilian_deaths'].sum():,}")

Panel rows: 11,868 (should be 11,868)

Outcome summary:
  Country-months with zero one-sided events: 9,305 (78.4%)
  Country-months with 1+ one-sided events:   2,563 (21.6%)

  Total one-sided events: 14,549
  Total one-sided deaths: 112,459
  Total civilian deaths:  130,611


## 4. Treatment Variable: UN Peacekeeping Missions

The treatment is whether a UN peacekeeping mission was active in a country during a given month. We construct this variable manually from the official UN Department of Peace Operations mission records.

Each mission has a known start date and end date (or is ongoing). We create two variables:

- `pk_present`: binary (0/1), was any UN peacekeeping mission active in this country-month?
- `pk_mission`: the mission name (for reference)

We focus on missions with a military/police component deployed to a specific country. Observer missions and political missions without armed personnel are excluded because they operate through different mechanisms than the troop deployment we want to study.

This approach is transparent and reproducible. Every researcher can verify the mission dates against UN records. It avoids dependency on external databases that may require registration or change format.

In [30]:
# UN Peacekeeping missions in Africa, 2000-2022
# Source: UN Department of Peace Operations
# Format: (mission_name, iso3, start_year, start_month, end_year, end_month)
# end_year=2022, end_month=12 means the mission was still active at end of our panel

pk_missions = [
    # Democratic Republic of Congo
    ('MONUC',    'COD', 1999, 11, 2010,  6),
    ('MONUSCO',  'COD', 2010,  7, 2022, 12),
    
    # Sudan / Darfur
    ('UNMIS',    'SDN', 2005,  3, 2011,  7),
    ('UNAMID',   'SDN', 2007,  7, 2020, 12),
    
    # South Sudan
    ('UNMISS',   'SSD', 2011,  7, 2022, 12),
    
    # Central African Republic
    ('MINURCA',  'CAF', 1998,  4, 2000,  2),
    ('MINUSCA',  'CAF', 2014,  9, 2022, 12),
    
    # Mali
    ('MINUSMA',  'MLI', 2013,  4, 2023,  6),  # ended Jun 2023, clamp to Dec 2022
    
    # Côte d'Ivoire
    ('UNOCI',    'CIV', 2004,  4, 2017,  6),
    
    # Liberia
    ('UNMIL',    'LBR', 2003,  9, 2018,  3),
    
    # Sierra Leone
    ('UNAMSIL',  'SLE', 1999, 10, 2005, 12),
    
    # Ethiopia/Eritrea
    ('UNMEE',    'ETH', 2000,  7, 2008,  7),
    ('UNMEE',    'ERI', 2000,  7, 2008,  7),
    
    # Burundi
    ('ONUB',     'BDI', 2004,  6, 2006, 12),
    
    # Western Sahara (Morocco)
    ('MINURSO',  'MAR', 1991,  4, 2022, 12),
    
    # Mozambique (ONUMOZ ended 1994, but small follow-on)
    # No major mission in our 2000-2022 window
    
    # Rwanda (UNAMIR ended 1996)
    # No major mission in our 2000-2022 window
    
    # Somalia (UNOSOM ended 1995; AMISOM is AU not UN)
    # AMISOM/ATMIS is African Union, not UN peacekeeping
    # We exclude it to keep the treatment clean: UN missions only
]

# Build a dataframe of mission-

In [31]:
print("Cell 18 starting...")

# UN Peacekeeping missions in Africa, 2000-2022
# Format: (mission_name, iso3, start_year, start_month, end_year, end_month)

pk_missions = [
    ('MONUC',    'COD', 1999, 11, 2010,  6),
    ('MONUSCO',  'COD', 2010,  7, 2022, 12),
    ('UNMIS',    'SDN', 2005,  3, 2011,  7),
    ('UNAMID',   'SDN', 2007,  7, 2020, 12),
    ('UNMISS',   'SSD', 2011,  7, 2022, 12),
    ('MINURCA',  'CAF', 1998,  4, 2000,  2),
    ('MINUSCA',  'CAF', 2014,  9, 2022, 12),
    ('MINUSMA',  'MLI', 2013,  4, 2022, 12),
    ('UNOCI',    'CIV', 2004,  4, 2017,  6),
    ('UNMIL',    'LBR', 2003,  9, 2018,  3),
    ('UNAMSIL',  'SLE', 1999, 10, 2005, 12),
    ('UNMEE',    'ETH', 2000,  7, 2008,  7),
    ('UNMEE',    'ERI', 2000,  7, 2008,  7),
    ('ONUB',     'BDI', 2004,  6, 2006, 12),
    ('MINURSO',  'MAR', 1991,  4, 2022, 12),
]

print(f"Missions defined: {len(pk_missions)}")

# Build mission-month rows
pk_rows = []
for mission, iso3, sy, sm, ey, em in pk_missions:
    if ey > 2022 or (ey == 2022 and em > 12):
        ey, em = 2022, 12
    if sy < 2000:
        sy, sm = 2000, 1
    start = pd.Timestamp(year=sy, month=sm, day=1)
    end = pd.Timestamp(year=ey, month=em, day=1)
    for date in pd.date_range(start, end, freq='MS'):
        pk_rows.append({
            'iso3': iso3,
            'year': date.year,
            'month': date.month,
            'pk_mission': mission
        })

print(f"Mission-month rows: {len(pk_rows)}")

pk_df = pd.DataFrame(pk_rows)

# Collapse to country-month: keep first mission name, mark as treated
pk_monthly = (pk_df
    .groupby(['iso3', 'year', 'month'])['pk_mission']
    .first()
    .reset_index()
)
pk_monthly['pk_present'] = 1

print(f"Peacekeeping country-months: {len(pk_monthly):,}")
print(f"Countries with PK missions: {pk_monthly['iso3'].nunique()}")

for iso3 in sorted(pk_monthly['iso3'].unique()):
    subset = pk_df[pk_df['iso3'] == iso3]
    missions = ', '.join(sorted(subset['pk_mission'].unique()))
    n_months = len(pk_monthly[pk_monthly['iso3'] == iso3])
    print(f"  {iso3}: {missions} ({n_months} months)")

Cell 18 starting...
Missions defined: 15
Mission-month rows: 1779
Peacekeeping country-months: 1,730
Countries with PK missions: 12
  BDI: ONUB (31 months)
  CAF: MINURCA, MINUSCA (102 months)
  CIV: UNOCI (159 months)
  COD: MONUC, MONUSCO (276 months)
  ERI: UNMEE (97 months)
  ETH: UNMEE (97 months)
  LBR: UNMIL (175 months)
  MAR: MINURSO (276 months)
  MLI: MINUSMA (117 months)
  SDN: UNAMID, UNMIS (190 months)
  SLE: UNAMSIL (72 months)
  SSD: UNMISS (138 months)


### 4.1 Merge Peacekeeping onto Panel

We left-join the peacekeeping data onto the panel. Country-months without a mission get `pk_present = 0`. This gives us the treated/untreated split that DML and Causal Forests will use.

In [32]:
# Merge peacekeeping onto panel
panel = panel.merge(pk_monthly, on=['iso3', 'year', 'month'], how='left')

# Fill NaN: no mission means pk_present = 0
panel['pk_present'] = panel['pk_present'].fillna(0).astype(int)
panel['pk_mission'] = panel['pk_mission'].fillna('None')

# Summary
treated = panel['pk_present'].sum()
untreated = len(panel) - treated
print(f"Treatment summary:")
print(f"  Treated (PK present):   {treated:,} country-months ({treated/len(panel):.1%})")
print(f"  Untreated (no PK):      {untreated:,} country-months ({untreated/len(panel):.1%})")

print(f"\nNaive comparison (this is BIASED by selection):")
treated_violence = panel[panel['pk_present'] == 1]['onesided_events'].mean()
untreated_violence = panel[panel['pk_present'] == 0]['onesided_events'].mean()
print(f"  Avg one-sided events WITH peacekeeping:    {treated_violence:.2f}")
print(f"  Avg one-sided events WITHOUT peacekeeping: {untreated_violence:.2f}")
print(f"  Naive difference: {treated_violence - untreated_violence:.2f}")
print(f"\n  (Positive difference = more violence with PK, reflecting selection bias)")

Treatment summary:
  Treated (PK present):   1,730 country-months (14.6%)
  Untreated (no PK):      10,138 country-months (85.4%)

Naive comparison (this is BIASED by selection):
  Avg one-sided events WITH peacekeeping:    3.70
  Avg one-sided events WITHOUT peacekeeping: 0.80
  Naive difference: 2.89

  (Positive difference = more violence with PK, reflecting selection bias)


## 5. Confounder: World Bank Development Indicators

GDP per capita and population are standard controls in cross-national conflict studies. GDP proxies for state capacity and economic development. Population proxies for country size and the pool of potential combatants and victims.

We use the `wbgapi` Python package to download these indicators directly from the World Bank API. The data is annual, so we merge it by country-year: all months within the same year get the same GDP and population value. This is standard practice because within-year GDP variation is not meaningful for this analysis.

In [33]:
import wbgapi as wb

In [34]:
# Download GDP per capita and population for African countries, 2000-2022
iso3_list_panel = sorted(panel['iso3'].unique())

# GDP per capita (constant 2015 USD)
print("Downloading GDP per capita...")
gdp_data = wb.data.DataFrame('NY.GDP.PCAP.KD', iso3_list_panel, range(2000, 2023))
gdp_data = gdp_data.reset_index()
gdp_data = gdp_data.rename(columns={'economy': 'iso3'})

# Reshape from wide to long
gdp_long = gdp_data.melt(id_vars='iso3', var_name='year_str', value_name='gdp_pc')
gdp_long['year'] = gdp_long['year_str'].str.replace('YR', '').astype(int)
gdp_long = gdp_long[['iso3', 'year', 'gdp_pc']]

print(f"GDP records: {len(gdp_long):,}")
print(f"Missing GDP values: {gdp_long['gdp_pc'].isna().sum():,}")

# Population
print("\nDownloading population...")
pop_data = wb.data.DataFrame('SP.POP.TOTL', iso3_list_panel, range(2000, 2023))
pop_data = pop_data.reset_index()
pop_data = pop_data.rename(columns={'economy': 'iso3'})

pop_long = pop_data.melt(id_vars='iso3', var_name='year_str', value_name='population')
pop_long['year'] = pop_long['year_str'].str.replace('YR', '').astype(int)
pop_long = pop_long[['iso3', 'year', 'population']]

print(f"Population records: {len(pop_long):,}")
print(f"Missing population values: {pop_long['population'].isna().sum():,}")

# Merge GDP and population
wb_data = gdp_long.merge(pop_long, on=['iso3', 'year'], how='outer')
print(f"\nWorld Bank data: {len(wb_data):,} country-years")
print(f"\nCountries with most missing GDP:")
missing_by_country = gdp_long[gdp_long['gdp_pc'].isna()].groupby('iso3').size().sort_values(ascending=False)
if len(missing_by_country) > 0:
    print(missing_by_country.head(10).to_string())
else:
    print("  None")

GDP records: 989
Missing GDP values: 39

Population records: 989
Missing population values: 0

World Bank data: 989 country-years

Countries with most missing GDP:
iso3
SSD    15
DJI    13
ERI    11


### 5.1 Merge World Bank Data onto Panel

World Bank data is annual. We merge by country and year, so all 12 months within a year get the same GDP and population value. Missing GDP values are forward-filled within each country: if 2003 is missing but 2002 and 2004 exist, we carry the 2002 value forward. This is a reasonable approach for slow-moving variables like GDP.

In [35]:
# Merge World Bank data onto panel by country-year
panel = panel.merge(wb_data, on=['iso3', 'year'], how='left')

# Check missingness before filling
print("Missing values before filling:")
print(f"  GDP per capita: {panel['gdp_pc'].isna().sum():,} ({panel['gdp_pc'].isna().mean():.1%})")
print(f"  Population:     {panel['population'].isna().sum():,} ({panel['population'].isna().mean():.1%})")

# Forward-fill within each country, then back-fill remaining
panel = panel.sort_values(['iso3', 'year', 'month'])
panel['gdp_pc'] = panel.groupby('iso3')['gdp_pc'].transform(lambda x: x.ffill().bfill())
panel['population'] = panel.groupby('iso3')['population'].transform(lambda x: x.ffill().bfill())

# Check missingness after filling
print("\nMissing values after filling:")
print(f"  GDP per capita: {panel['gdp_pc'].isna().sum():,} ({panel['gdp_pc'].isna().mean():.1%})")
print(f"  Population:     {panel['population'].isna().sum():,} ({panel['population'].isna().mean():.1%})")

# Log-transform for modeling (GDP and population are right-skewed)
panel['ln_gdp_pc'] = np.log(panel['gdp_pc'])
panel['ln_population'] = np.log(panel['population'])

print("\nGDP per capita summary (constant 2015 USD):")
print(panel.groupby('iso3')['gdp_pc'].mean().describe().to_string())

Missing values before filling:
  GDP per capita: 468 (3.9%)
  Population:     0 (0.0%)

Missing values after filling:
  GDP per capita: 0 (0.0%)
  Population:     0 (0.0%)

GDP per capita summary (constant 2015 USD):
count       43.000000
mean      1658.920427
std       1803.637515
min        259.333091
25%        639.527821
50%        981.926242
75%       1909.202697
max      10146.463011


## 6. Confounder: Active Armed Conflict

A key confounder is whether a country has active armed conflict. Countries at war are more likely to receive peacekeeping AND more likely to have violence against civilians. Without controlling for this, we confuse the effect of peacekeeping with the effect of being in a war zone.

We derive this from the UCDP GED data we already loaded. State-based conflict events (type_of_violence == 1) indicate fighting between government forces and organized armed groups. We create two variables:

- `active_conflict`: binary, did this country have any state-based conflict events in the past 12 months?
- `battle_events_12m`: count of state-based events in the past 12 months (a continuous measure of conflict intensity)

Using a 12-month rolling window rather than the calendar year avoids arbitrary boundary effects and captures conflict dynamics more precisely.

In [36]:
# State-based conflict events from GED
statebased = ged_africa[ged_africa['type_of_violence'] == 1].copy()

# Aggregate to country-month
statebased_agg = (statebased
    .groupby(['iso3', 'year', 'month'])
    .agg(battle_events=('id', 'count'))
    .reset_index()
)

# Merge onto panel
panel = panel.merge(statebased_agg, on=['iso3', 'year', 'month'], how='left')
panel['battle_events'] = panel['battle_events'].fillna(0).astype(int)

# Calculate rolling 12-month sum of battle events per country
panel = panel.sort_values(['iso3', 'year', 'month'])
panel['battle_events_12m'] = (panel
    .groupby('iso3')['battle_events']
    .transform(lambda x: x.rolling(12, min_periods=1).sum())
)

# Binary: any state-based conflict in past 12 months
panel['active_conflict'] = (panel['battle_events_12m'] > 0).astype(int)

print("Active conflict summary:")
print(f"  Country-months with active conflict: {panel['active_conflict'].sum():,} ({panel['active_conflict'].mean():.1%})")
print(f"  Country-months without: {(panel['active_conflict'] == 0).sum():,} ({(panel['active_conflict'] == 0).mean():.1%})")

print("\nCountries with most months of active conflict:")
conflict_by_country = (panel
    .groupby('iso3')['active_conflict']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
for iso3, months in conflict_by_country.items():
    name = iso3_to_name.get(iso3, iso3)
    print(f"  {name}: {months:.0f} months ({months/276:.0%} of panel)")

Active conflict summary:
  Country-months with active conflict: 4,698 (39.6%)
  Country-months without: 7,170 (60.4%)

Countries with most months of active conflict:
  Algeria: 276 months (100% of panel)
  Ethiopia: 276 months (100% of panel)
  Sudan: 275 months (100% of panel)
  Angola: 258 months (93% of panel)
  DR Congo (Zaire): 252 months (91% of panel)
  Burundi: 247 months (89% of panel)
  Somalia: 237 months (86% of panel)
  Chad: 221 months (80% of panel)
  Central African Republic: 212 months (77% of panel)
  Mali: 200 months (72% of panel)


## 7. Confounder: Ethnic Fractionalization

Ethnic fractionalization measures how ethnically divided a country is. It ranges from 0 (completely homogeneous) to 1 (every person belongs to a different group). This matters because ethnically fragmented countries tend to have different patterns of civilian violence, and the effectiveness of peacekeeping may vary with ethnic composition.

The standard source is the Ethnic Power Relations (EPR) dataset from ETH Zurich. Downloading and processing EPR requires several steps. For this project, we use a widely cited static measure: the ethnic fractionalization index from Alesina et al. (2003), which covers virtually all countries. This index does not change over time, which is appropriate because ethnic composition shifts very slowly. We code it as a time-invariant country-level variable.

In [37]:
# Ethnic fractionalization index (Alesina et al., 2003)
# Source: widely used in conflict studies, values from the original paper
# This is a static measure: one value per country

ethnic_frac = {
    'DZA': 0.34, 'AGO': 0.79, 'BEN': 0.79, 'BFA': 0.74, 'BDI': 0.30,
    'CMR': 0.86, 'CAF': 0.83, 'TCD': 0.86, 'COG': 0.87, 'COD': 0.87,
    'DJI': 0.80, 'ERI': 0.65, 'ETH': 0.72, 'GMB': 0.79, 'GHA': 0.67,
    'GIN': 0.74, 'GNB': 0.81, 'CIV': 0.82, 'KEN': 0.86, 'SWZ': 0.06,
    'LBR': 0.91, 'LBY': 0.08, 'MDG': 0.06, 'MLI': 0.69, 'MRT': 0.62,
    'MAR': 0.48, 'MOZ': 0.69, 'NAM': 0.63, 'NER': 0.74, 'NGA': 0.85,
    'RWA': 0.32, 'SEN': 0.69, 'SLE': 0.82, 'SOM': 0.81, 'ZAF': 0.75,
    'SSD': 0.77, 'SDN': 0.71, 'TZA': 0.74, 'TGO': 0.71, 'TUN': 0.04,
    'UGA': 0.93, 'ZMB': 0.78, 'ZWE': 0.39
}

# Map onto panel
panel['ethnic_frac'] = panel['iso3'].map(ethnic_frac)

# Check for missing
missing = panel[panel['ethnic_frac'].isna()]['iso3'].unique()
if len(missing) > 0:
    print(f"WARNING: missing ethnic_frac for {missing}")
else:
    print("All 43 countries have ethnic fractionalization values")

print(f"\nEthnic fractionalization summary:")
country_ef = panel.groupby('iso3')['ethnic_frac'].first().sort_values()
print(f"  Lowest:  {country_ef.index[0]} ({iso3_to_name[country_ef.index[0]]}): {country_ef.iloc[0]:.2f}")
print(f"  Median:  {country_ef.median():.2f}")
print(f"  Highest: {country_ef.index[-1]} ({iso3_to_name[country_ef.index[-1]]}): {country_ef.iloc[-1]:.2f}")

All 43 countries have ethnic fractionalization values

Ethnic fractionalization summary:
  Lowest:  TUN (Tunisia): 0.04
  Median:  0.74
  Highest: UGA (Uganda): 0.93


## 8. Lagged Variables and Final Panel

Causal effects take time. Peacekeepers do not reduce violence the month they arrive. Many studies use a one-month lag between treatment and outcome: does peacekeeping presence in month t predict violence in month t+1?

We create lagged versions of the treatment and outcome variables. We also create a lagged outcome as a confounder: prior violence levels predict both future peacekeeping decisions and future violence, making it a classic confounder.

After adding lags, we drop the first month of each country's series (January 2000) because the lag is undefined. The final panel starts from February 2000.

In [38]:
# Sort to ensure correct lag order
panel = panel.sort_values(['iso3', 'year', 'month']).reset_index(drop=True)

# Lagged treatment: peacekeeping presence last month
panel['pk_present_lag1'] = panel.groupby('iso3')['pk_present'].shift(1)

# Lagged outcome: one-sided violence last month
panel['onesided_events_lag1'] = panel.groupby('iso3')['onesided_events'].shift(1)

# Lagged battle events: conflict intensity last month
panel['battle_events_lag1'] = panel.groupby('iso3')['battle_events'].shift(1)

# Drop rows where lags are undefined (first month per country)
rows_before = len(panel)
panel = panel.dropna(subset=['pk_present_lag1']).reset_index(drop=True)
rows_after = len(panel)

# Convert lag columns to int
panel['pk_present_lag1'] = panel['pk_present_lag1'].astype(int)
panel['onesided_events_lag1'] = panel['onesided_events_lag1'].astype(int)
panel['battle_events_lag1'] = panel['battle_events_lag1'].astype(int)

print(f"Rows dropped (first month per country): {rows_before - rows_after}")
print(f"Final panel: {len(panel):,} country-months")
print(f"Date range: {panel['date'].min().strftime('%Y-%m')} to {panel['date'].max().strftime('%Y-%m')}")

print(f"\nColumn list:")
for col in panel.columns:
    non_null = panel[col].notna().sum()
    print(f"  {col}: {non_null:,} non-null ({panel[col].dtype})")

Rows dropped (first month per country): 43
Final panel: 11,825 country-months
Date range: 2000-02 to 2022-12

Column list:
  iso3: 11,825 non-null (str)
  year: 11,825 non-null (int64)
  month: 11,825 non-null (int64)
  date: 11,825 non-null (datetime64[us])
  country_name: 11,825 non-null (str)
  onesided_events: 11,825 non-null (int64)
  onesided_deaths: 11,825 non-null (int64)
  civilian_deaths: 11,825 non-null (int64)
  all_events: 11,825 non-null (int64)
  pk_mission: 11,825 non-null (str)
  pk_present: 11,825 non-null (int64)
  gdp_pc: 11,825 non-null (float64)
  population: 11,825 non-null (float64)
  ln_gdp_pc: 11,825 non-null (float64)
  ln_population: 11,825 non-null (float64)
  battle_events: 11,825 non-null (int64)
  battle_events_12m: 11,825 non-null (float64)
  active_conflict: 11,825 non-null (int64)
  ethnic_frac: 11,825 non-null (float64)
  pk_present_lag1: 11,825 non-null (int64)
  onesided_events_lag1: 11,825 non-null (int64)
  battle_events_lag1: 11,825 non-null (in

## 9. Save the Final Panel

We save the completed panel to `data/processed/panel_country_month.csv`. This file is the input for all remaining notebooks. Every row is one country-month with outcome, treatment, and confounder variables ready for causal estimation.

In [39]:
# Save to processed data folder
output_path = 'data/processed/panel_country_month.csv'
panel.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"Panel saved to: {output_path}")
print(f"File size: {file_size:.1f} MB")
print(f"Dimensions: {panel.shape[0]:,} rows × {panel.shape[1]} columns")

print(f"\nFinal summary:")
print(f"  Countries: {panel['iso3'].nunique()}")
print(f"  Date range: {panel['date'].min().strftime('%Y-%m')} to {panel['date'].max().strftime('%Y-%m')}")
print(f"  Treated country-months: {panel['pk_present'].sum():,} ({panel['pk_present'].mean():.1%})")
print(f"  One-sided violence events: {panel['onesided_events'].sum():,}")
print(f"  Missing values: {panel.isna().sum().sum()}")

print(f"\nVariables for causal estimation:")
print(f"  Outcome (Y):     onesided_events, onesided_deaths, civilian_deaths")
print(f"  Treatment (T):   pk_present (or pk_present_lag1)")
print(f"  Confounders (W): ln_gdp_pc, ln_population, ethnic_frac,")
print(f"                   active_conflict, battle_events_12m,")
print(f"                   onesided_events_lag1, battle_events_lag1")

Panel saved to: data/processed/panel_country_month.csv
File size: 1.5 MB
Dimensions: 11,825 rows × 22 columns

Final summary:
  Countries: 43
  Date range: 2000-02 to 2022-12
  Treated country-months: 1,726 (14.6%)
  One-sided violence events: 14,500
  Missing values: 0

Variables for causal estimation:
  Outcome (Y):     onesided_events, onesided_deaths, civilian_deaths
  Treatment (T):   pk_present (or pk_present_lag1)
  Confounders (W): ln_gdp_pc, ln_population, ethnic_frac,
                   active_conflict, battle_events_12m,
                   onesided_events_lag1, battle_events_lag1
